# Kill Counter RMSE Accuracy Report

This notebook compares human-coded kill counter quantities (`Y`) with OCR/calculated kill counter quantities (`Yhat`) using the 5-second interval CSV.

For this run: `VIDEO_NAME = "video4_Imelda_100"`, human sheet = `GemCalculator(Video4)`, human column = `Kill counter`.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import base64, html, math, re
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 80)


## Files

Change only these paths if you run a different video later.


In [ ]:
def find_project_dir():
    """Find the kill-counter project folder after MNL is organized."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if base.name == '01_kill_counter_and_time_stamp':
            return base
        child = base / '01_kill_counter_and_time_stamp'
        if child.exists() and child.is_dir():
            return child
        mnl_child = base / 'MNL' / '01_kill_counter_and_time_stamp'
        if mnl_child.exists() and mnl_child.is_dir():
            return mnl_child
    return Path.cwd()


PROJECT_DIR = find_project_dir()
MNL_DIR = PROJECT_DIR.parent if PROJECT_DIR.name == '01_kill_counter_and_time_stamp' else PROJECT_DIR
HUMAN_EXCEL_DIR = MNL_DIR / '00-Human Coded Sheet'
OUTPUT_ROOT = PROJECT_DIR / 'results'


def resolve_path(path):
    p = Path(path).expanduser()
    candidates = [
        p,
        Path.cwd() / p,
        PROJECT_DIR / p,
        PROJECT_DIR / p.name,
        MNL_DIR / p,
        MNL_DIR / p.name,
        HUMAN_EXCEL_DIR / p.name,
        OUTPUT_ROOT / p.name,
        Path.cwd() / 'MNL' / p.name,
        Path.cwd() / 'MNL' / '00-Human Coded Sheet' / p.name,
        Path.cwd() / 'MNL' / '01_kill_counter_and_time_stamp' / p,
        Path.cwd().parent / p.name,
        Path.cwd().parent / '00-Human Coded Sheet' / p.name,
        Path.cwd().parent / 'MNL' / '00-Human Coded Sheet' / p.name,
        Path.cwd().parent / 'MNL' / '01_kill_counter_and_time_stamp' / p,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return p


def clean_folder_label(text):
    label = re.sub(r'[^A-Za-z0-9._-]+', '_', str(text)).strip('_')
    return label or 'video'


def clean_file_label(text):
    label = Path(str(text)).stem.lower()
    label = re.sub(r'[^a-z0-9]+', '_', label).strip('_')
    return label or 'video'


HUMAN_XLSX = resolve_path('Gaming Content Videos.xlsx')
HUMAN_SHEET = "GemCalculator(Video4)"
HUMAN_TIME_COLUMN = "Time Stamp"
HUMAN_KILL_COLUMN = "Kill counter"
VIDEO_NAME = "video4_Imelda_100"

VIDEO_OUTPUT_DIR = OUTPUT_ROOT / clean_folder_label(VIDEO_NAME)
VIDEO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

machine_guess = VIDEO_OUTPUT_DIR / f"kill_time_{clean_file_label(VIDEO_NAME)}_full_5sec_intervals.csv"
legacy_machine_guess = resolve_path(f"kill_time_{clean_file_label(VIDEO_NAME)}_full_5sec_intervals.csv")
MACHINE_CSV = machine_guess if machine_guess.exists() else legacy_machine_guess

OUT_DIR = VIDEO_OUTPUT_DIR
COMPARISON_CSV = OUT_DIR / "kill_counter_rmse_comparison.csv"
SUMMARY_CSV = OUT_DIR / "kill_counter_rmse_summary.csv"
PLOT_PNG = OUT_DIR / "kill_counter_rmse_plot.png"
REPORT_HTML = OUT_DIR / "kill_counter_rmse_report.html"
WORST10_CSV = OUT_DIR / "kill_counter_rmse_worst10_intervals.csv"
WORST10_TSV = OUT_DIR / "kill_counter_rmse_worst10_intervals.tsv"
WORST10_MD = OUT_DIR / "kill_counter_rmse_worst10_intervals.md"

print(f"Project folder: {PROJECT_DIR}")
print(f"Video output folder: {VIDEO_OUTPUT_DIR}")
print(f"Using human workbook: {HUMAN_XLSX}")
print(f"Using human sheet: {HUMAN_SHEET}")
print(f"Using human kill column: {HUMAN_KILL_COLUMN}")
print(f"Using OCR interval CSV: {MACHINE_CSV}")


## Load human Excel and OCR interval CSV

The Excel reader below avoids `openpyxl`, so this notebook works with the packages already on this machine.


In [ ]:
NS = {
    "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
    "rel": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
}

def col_index(cell_ref):
    n = 0
    for ch in ''.join(c for c in cell_ref if c.isalpha()):
        n = n * 26 + ord(ch.upper()) - 64
    return n - 1

def read_xlsx_sheet(path, sheet_name):
    with ZipFile(path) as z:
        shared = []
        if "xl/sharedStrings.xml" in z.namelist():
            root = ET.fromstring(z.read("xl/sharedStrings.xml"))
            shared = [''.join(t.text or '' for t in si.findall('.//main:t', NS)) for si in root.findall('main:si', NS)]
        wb = ET.fromstring(z.read("xl/workbook.xml"))
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        relmap = {r.attrib["Id"]: r.attrib["Target"] for r in rels}
        sheets = wb.findall('main:sheets/main:sheet', NS)
        match = next(s for s in sheets if s.attrib["name"] == sheet_name)
        rid = match.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]
        target = relmap[rid].lstrip('/')
        sheet_xml = target if target.startswith('xl/') else 'xl/' + target
        root = ET.fromstring(z.read(sheet_xml))
        rows = []
        for row in root.findall('.//main:sheetData/main:row', NS):
            values = []
            for cell in row.findall('main:c', NS):
                while len(values) <= col_index(cell.attrib['r']):
                    values.append('')
                v = cell.find('main:v', NS)
                text = '' if v is None else (v.text or '')
                if cell.attrib.get('t') == 's' and text:
                    text = shared[int(text)]
                values[col_index(cell.attrib['r'])] = text
            rows.append(values)
    header = rows[0]
    body = []
    for row in rows[1:]:
        if len(row) < len(header):
            row = row + [''] * (len(header) - len(row))
        body.append(row[:len(header)])
    return pd.DataFrame(body, columns=header)

human = read_xlsx_sheet(HUMAN_XLSX, HUMAN_SHEET)[[HUMAN_TIME_COLUMN, HUMAN_KILL_COLUMN]]
ocr = pd.read_csv(MACHINE_CSV)[["Time Stamp", "Matched Video Second", "Source OCR Time Stamp", "Kill Counter Quantity"]]

human = human.rename(columns={HUMAN_TIME_COLUMN: "Time Stamp", HUMAN_KILL_COLUMN: "Y"})
ocr = ocr.rename(columns={"Kill Counter Quantity": "Yhat"})

for df in [human, ocr]:
    df["Time Stamp"] = df["Time Stamp"].astype(str).str.replace(r"\s+", "", regex=True)

human["Y"] = pd.to_numeric(human["Y"], errors="coerce")
ocr["Yhat"] = pd.to_numeric(ocr["Yhat"], errors="coerce")
human = human.dropna(subset=["Time Stamp", "Y"])
human = human[human["Time Stamp"].str.contains("-", na=False)].copy()
ocr = ocr.dropna(subset=["Time Stamp", "Yhat"])

print(f"Human intervals: {len(human)}")
print(f"OCR intervals: {len(ocr)}")
display(human.head())
display(ocr.head())


## RMSE calculation


In [ ]:
comparison = human.merge(ocr, on="Time Stamp", how="inner")
comparison["Error"] = comparison["Yhat"] - comparison["Y"]
comparison["Absolute Error"] = comparison["Error"].abs()
comparison["Squared Error"] = comparison["Error"] ** 2

rmse = math.sqrt(comparison["Squared Error"].mean())
summary = pd.DataFrame({
    "Metric": [
        "Human intervals", "OCR intervals", "Matched intervals", "RMSE", "MAE", "Bias (Yhat - Y)",
        "Max absolute error", "Exact matches", "Percent exact", "Percent within 1 kill", "Percent within 5 kills"
    ],
    "Value": [
        len(human), len(ocr), len(comparison), rmse, comparison["Absolute Error"].mean(), comparison["Error"].mean(),
        comparison["Absolute Error"].max(), int((comparison["Error"] == 0).sum()),
        (comparison["Error"] == 0).mean(), (comparison["Absolute Error"] <= 1).mean(), (comparison["Absolute Error"] <= 5).mean()
    ]
})

worst = comparison.sort_values("Absolute Error", ascending=False).head(10)
worst_cols = ["Time Stamp", "Y", "Yhat", "Error", "Absolute Error", "Matched Video Second", "Source OCR Time Stamp"]

OUT_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(COMPARISON_CSV, index=False)
summary.to_csv(SUMMARY_CSV, index=False)
worst[worst_cols].to_csv(WORST10_CSV, index=False)
worst[worst_cols].to_csv(WORST10_TSV, sep="	", index=False)

md_rows = [
    "# Worst 10 Kill Counter Intervals",
    "",
    "| Time Stamp | Human Y | OCR Yhat | Error | Absolute Error | Matched Video Second | Source OCR Time Stamp |",
    "|---|---:|---:|---:|---:|---:|---|",
]
for _, row in worst[worst_cols].iterrows():
    source_ts = "" if pd.isna(row["Source OCR Time Stamp"]) else str(row["Source OCR Time Stamp"])
    md_rows.append(
        f"| {row['Time Stamp']} | {row['Y']:.0f} | {row['Yhat']:.0f} | {row['Error']:.0f} | "
        f"{row['Absolute Error']:.0f} | {int(row['Matched Video Second'])} | {source_ts} |"
    )
line_break = chr(10)
WORST10_MD.write_text(line_break.join(md_rows) + line_break, encoding="utf-8")

display(summary)
print(f"RMSE = {rmse:.4f} kills")
print(f"Saved comparison CSV: {COMPARISON_CSV}")
print(f"Saved summary CSV: {SUMMARY_CSV}")
print(f"Saved worst-10 CSV: {WORST10_CSV}")


## Plot and saved report


In [ ]:
plot_df = comparison.reset_index(drop=True)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(plot_df.index, plot_df["Y"], label="Human Y", linewidth=2)
axes[0].plot(plot_df.index, plot_df["Yhat"], label="OCR Yhat", linewidth=2, alpha=0.85)
axes[0].set_title(f"Kill Counter by 5-Second Interval (RMSE = {rmse:.2f})")
axes[0].set_ylabel("Kill Counter Quantity")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].bar(plot_df.index, plot_df["Error"], color=np.where(plot_df["Error"] >= 0, "#2F80ED", "#EB5757"))
axes[1].axhline(rmse, color="black", linestyle="--", linewidth=1, label="RMSE")
axes[1].axhline(-rmse, color="black", linestyle="--", linewidth=1)
axes[1].set_ylabel("Yhat - Y")
axes[1].set_xlabel("5-second interval row")
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
fig.savefig(PLOT_PNG, dpi=180, bbox_inches="tight")
plt.show()

metrics = {row["Metric"]: row["Value"] for _, row in summary.iterrows()}
image_b64 = base64.b64encode(PLOT_PNG.read_bytes()).decode("ascii")

report = f'''
<html><head><meta charset="utf-8"><title>Kill Counter RMSE Report</title>
<style>body{{font-family:Arial,sans-serif;max-width:1000px;margin:32px auto;line-height:1.45}} table{{border-collapse:collapse}} td,th{{border:1px solid #ddd;padding:6px 8px}} th{{background:#f3f3f3}} img{{max-width:100%}}</style>
</head><body>
<h1>Kill Counter RMSE Report - {html.escape(VIDEO_NAME)}</h1>
<p><b>Method:</b> Human-coded kill counter values (<code>Y</code>) were merged with OCR interval values (<code>Yhat</code>) by the exact 5-second <code>Time Stamp</code>.</p>
<p><b>Formula:</b> RMSE = sqrt(mean((Yhat - Y)^2)).</p>
<h2>Results</h2>
<ul>
<li><b>Matched intervals:</b> {int(metrics['Matched intervals'])}</li>
<li><b>RMSE:</b> {metrics['RMSE']:.4f} kills</li>
<li><b>MAE:</b> {metrics['MAE']:.4f} kills</li>
<li><b>Bias (Yhat - Y):</b> {metrics['Bias (Yhat - Y)']:.4f} kills</li>
<li><b>Max absolute error:</b> {metrics['Max absolute error']:.0f} kills</li>
<li><b>Percent exact:</b> {metrics['Percent exact']:.1%}</li>
<li><b>Percent within 1 kill:</b> {metrics['Percent within 1 kill']:.1%}</li>
<li><b>Percent within 5 kills:</b> {metrics['Percent within 5 kills']:.1%}</li>
</ul>
<h2>RMSE Plot</h2>
<img src="data:image/png;base64,{image_b64}" alt="RMSE plot">
<h2>Worst 10 Intervals</h2>
{worst[worst_cols].to_html(index=False)}
<h2>Saved Files</h2>
<ul><li>{html.escape(str(COMPARISON_CSV))}</li><li>{html.escape(str(SUMMARY_CSV))}</li><li>{html.escape(str(PLOT_PNG))}</li><li>{html.escape(str(WORST10_CSV))}</li></ul>
</body></html>
'''
REPORT_HTML.write_text(report, encoding="utf-8")

print(f"Saved RMSE plot: {PLOT_PNG}")
print(f"Saved HTML report: {REPORT_HTML}")


## Latest Saved Results

Video: `video4_Imelda_100`  
Human sheet: `GemCalculator(Video4)`  
Human kill column: `Kill counter`

- Human intervals with values: 60
- OCR intervals: 240
- Matched intervals: 60
- RMSE: 2.9749 kills
- MAE: 1.6833 kills
- Bias (Yhat - Y): 0.6167 kills
- Max absolute error: 11 kills
- Percent exact: 41.7%
- Percent within 1 kill: 70.0%
- Percent within 5 kills: 91.7%

Saved folder: `MNL/01_kill_counter_and_time_stamp/results/video4_Imelda_100/`  
Saved report: `MNL/01_kill_counter_and_time_stamp/results/video4_Imelda_100/kill_counter_rmse_report.html`  
Saved RMSE plot: `MNL/01_kill_counter_and_time_stamp/results/video4_Imelda_100/kill_counter_rmse_plot.png`  
Saved worst-10 table: `MNL/01_kill_counter_and_time_stamp/results/video4_Imelda_100/kill_counter_rmse_worst10_intervals.csv`
